# Working with the Anthropic API

This notebook introduces the fundamentals of working with the **Anthropic Python SDK** and the **Messages API**.

We will start with a simple API request and progressively build toward a small conversational application. Along the way, we will explore how to:

* authenticate and initialize the Anthropic client
* send messages to Claude
* structure multi-turn conversations
* maintain conversation history
* control the length and structure of model responses
* use assistant-message prefilling
* handle model responses safely

The goal is not only to learn how to make an API call, but to understand the basic patterns used when building applications with Claude.


##  Setup  

Before making our first request to Claude, we need to prepare the Python environment and initialize the Anthropic client.

This section covers three steps:

- **Import the Anthropic SDK**: provides the Python interface for interacting with the Anthropic API.  
- **Load the API credentials**: retrieves the API key from the environment rather than storing it directly in the notebook.  
- **Initialize the client**: creates the Anthropic client that we will use throughout the notebook.  


### Import SDK  
 The anthropic package provides the Python SDK for interacting with Claude through the Anthropic API. 

In [1]:
from anthropic import Anthropic

### Load credentials  
  
This repository uses environment variables to keep the Anthropic API key separate from the code. The provided helper function (`helper.py`) loads the variables from the local `.env` file (see `.env_example` file).  

In [2]:
# Load environment variables ANTHROPIC_API_KEY
from helper import load_env
load_env()

### Initialize client  

Once the environment variables have been loaded, create an Anthropic client. The client will use the ANTHROPIC_API_KEY environment variable when making API requests.

We can now use client throughout the notebook to interact with Claude.

In [3]:
# Initialize the Anthropic client
client = Anthropic()

## Select a model  

The Anthropic API provides access to [different Claude models](https://platform.claude.com/docs/en/models/overview), each designed for different types of tasks and workloads.  

The model identifier is stored in a variable so that it can be reused throughout the notebook.

In [4]:
# Example model configuration.
MODEL_NAME="claude-sonnet-5"  

 **Note:** Model identifiers change over time. Use a model that is currently
 available to your Anthropic account and update `MODEL_NAME` if necessary.

## Making an API call  

The `messages.create()` method is the core API call. It accepts several paramenters that are shown in the following table:  


| Parameter    | Purpose                                  |
| ------------ | ---------------------------------------- |
| `model`      | Selects the Claude model                 |
| `max_tokens` | Sets the maximum number of output tokens |
| `messages`   | Contains the conversation                |

  
_Note_: Each message includes a role and content.  

### Make a request  

Let's start with a simple prompt and ask Claude to write a haiku about Anthropic.

In [8]:
# A basic prompt request: ask Claude to write a haiku
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a haiku about Anthropic"}
    ]
)

#print(response.content[0].text)

### Inspecting the complete response  

- For a simple text response, the generated text can be accessed directly from the first content block:

In [9]:
print(response.content[0].text)

Here's a haiku about Anthropic:

Careful hands shaping
minds that learn to think and pause—
safety, line by line


- If you want to explore the structure returned by the API, you can inspect the complete response object:

In [10]:
response

Message(id='msg_011CfkdehLuKeZ4hiR2qzJje', content=[TextBlock(citations=None, text="Here's a haiku about Anthropic:\n\nCareful hands shaping\nminds that learn to think and pause—\nsafety, line by line", type='text')], model='claude-sonnet-5', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=18, output_tokens=44, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

**Explanation output** : As it can been seen, the response is a structured object containing metadata, such as the message ID, and content blocks. In this notebook, we print only the blocks that actually contain text. This is a safe pattern, because the SDK may return other block types depending on model behavior and API updates.

> _Note_: This can be useful when learning the SDK because the response contains more information than just the generated text.

## The Messages Format  

Anthropic's Messages API uses a list of messages, where each message is a dictionary with:  

- `role`: either "user" or "assistant"  
- `content`: the text or structured content for that turn 

| Role | Meaning |
|------|---------|
| `user` | A message from the user/application |
| `assistant` | A previous response from Claude |


This format is fundamental for multi-turn conversations.  

The Messages API is stateless at the request level. To continue a conversation, the application sends previous messages again as part of the `messages` list.

 This example shows how to keep conversation context across turns.

In [11]:
# The assistant is told to respond in Dutch.
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "I am learning about Anthropic. Please answer in Dutch."},
        {"role": "assistant", "content": "Natuurlijk! Ik zal in het Nederlands antwoorden."},
        {"role": "user", "content": "What is Anthropic?"}
    ]
)


# The response object contains content blocks.
# A robust way to print the generated text:
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)


**Anthropic** is een Amerikaans AI-veiligheidsbedrijf dat werd opgericht in 2021 door Dario Amodei en Daniela Amodei, samen met enkele andere voormalige medewerkers van OpenAI. Het bedrijf is gevestigd in San Francisco, Californië.

**Belangrijke kenmerken van Anthropic:**

1. **Missie**: Anthropic richt zich op het ontwikkelen van veilige en betrouwbare kunstmatige intelligentie (AI). Het bedrijf legt sterk de nadruk op AI-veiligheid en ethiek.

2. **Claude**: Anthropic is vooral bekend om hun AI-assistent genaamd **Claude**, die vergelijkbaar is met ChatGPT van OpenAI. Claude wordt gebruikt voor tekstgeneratie, het beantwoorden van vragen, programmeren, en nog veel meer.

3. **Onderzoeksfocus**: Het bedrijf doet veel onderzoek naar hoe je AI-systemen kunt bouwen die nuttig, eerlijk en onschadelijk zijn (vaak aangeduid als "Helpful, Honest, and Harmless" of HHH).

4. **Constitutional AI**: Anthropic heeft een methode ontwikkeld genaamd "Constitutional AI", waarbij AI-modellen worden g

This example shows how conversation context is preserved. Once the assistant has responded earlier, the next user message can be interpreted in context.

##  Building a simple chatbot  

A chatbot can be built by storing the conversation history in a list of messages. Each time the user enters a new message, it is appended to the list, and the model generates a reply based on the whole conversation so far.  


### From individual requests to a conversation

So far, each API call has been created manually. We can now combine what we have learned to build a simple conversational loop.

The key idea is:

1. Get input from the user.
2. Add it to the message history.
3. Send the history to Claude.
4. Add Claude's response to the history.
5. Repeat.

In [12]:
print("Simple Chatbot (type 'quit' to exit)")
# Store conversation history
messages = []
while True:
    # Get user input
    user_input = input("You: ")
    # Check for quit command
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    # Add the user's latest message to the conversation history
    messages.append({"role": "user", "content": user_input})
    try:
        # Get response from Claude
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=200,
            messages=messages
        )
        # Extract and print Claude's response
        assistant_message = response.content[0].text
        print("Assistant:", assistant_message)
        
        # Store the assistant response so the model can continue the exchange
        messages.append({"role": "assistant", "content": assistant_message})
        
    except Exception as e:
        print(f"An error occurred: {e}")

Simple Chatbot (type 'quit' to exit)
Assistant: # About Anthropic

Anthropic is an AI safety company that I (Claude) was created by. Here are the key facts:

## Overview
- Founded in **2021** by former members of OpenAI, including siblings **Dario Amodei** (CEO) and **Daniela Amodei** (President)
- Headquartered in **San Francisco, California**
- Mission-focused on ensuring AI systems are **safe, beneficial, and understandable** as they become more capable

## Core Philosophy
Anthropic was founded on the belief that AI will be one of the most impactful technologies in history, and that careful, safety-foc
Goodbye!


This pattern is common in real-world chatbot applications, because it allows the model to remember prior turns and respond in a consistent style.

## Prefilling the Assistant Response  

In some situations, you may want to guide the assistant by providing a pre-written assistant message. This is useful when continuing a draft, shaping the tone, or maintaining a specific expected direction.

In [13]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a short poem about Dutch people"},
        {"role": "assistant", "content": "Oink"},
        {"role": "user", "content": "Continue the poem in a more polished, poetic style."}
    ]
)

#print(response.content[0].text)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

*Ahem — let me actually try that properly:*

**Lowlands**

Where rivers bow to human will,
and dikes hold back the sea's old claim,
the windmills turn against the chill,
slow sentinels without a name.

Bicycles hum on cobbled streets,
past gabled homes in evening gold,
while tulip fields in color greet
the mist that spring has gently rolled.

They speak in vowels round and plain,
and laugh with voices clear and dry,
a people shaped by wind and rain,
who built their ground beneath the sky.

In Delft blue light and canal's gleam,
in cheese and clog and painter's hand,
the Dutch have turned the water's dream
into a quiet, steadfast land.


**Explanation output** This example shows that an assistant message does not always have to be generated by the model in the same call. It can be included explicitly to steer the next output.  

## Model Parameters    

The API supports several generation parameters. Some of the most important ones are:  

- `max_tokens`  
- `stop_sequences`  
- other model-specific generation controls  

These parameters help us guide the model's output length and shape.

### Max Tokens  

`max_tokens` limits the total length of the model's output. It is one of the simplest ways to keep responses concise or allow longer output when needed.

In [14]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Write me an essay on LLMs"},
    ]
)
print(response.content[0].text)

# Large Language Models: Understanding the Technology Reshaping Human-Computer Interaction

## Introduction

Large Language Models (LLMs) have emerged as one of the most transformative technologies of the 21st century, fundamentally changing how humans interact with computers and how machines process and generate human language. From ch


In [15]:
response

Message(id='msg_011Cfkdr8jw7CGeNPoKdEWfA', content=[TextBlock(citations=None, text='# Large Language Models: Understanding the Technology Reshaping Human-Computer Interaction\n\n## Introduction\n\nLarge Language Models (LLMs) have emerged as one of the most transformative technologies of the 21st century, fundamentally changing how humans interact with computers and how machines process and generate human language. From ch', type='text')], model='claude-sonnet-5', role='assistant', stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=17, output_tokens=100, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

### Stop Sequences  

`stop_sequences` tells the model to stop generating once it encounters a specific string. This is especially useful when we want structured output, such as a numbered list that should terminate at a certain point.

In [16]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

# Technical Topics for Working on LLMs

## Foundations

1. **Linear Algebra** - vectors, matrices, eigenvalues, matrix decomposition
2. **Probability & Statistics** - distributions, Bayesian inference, information theory (entropy, KL divergence)
3. **Calculus & Optimization** - gradients, partial derivatives, convex/non-convex optimization
4. **Python Programming** - NumPy, PyTorch/TensorFlow, debugging large codebases
5. **Data Structures & Algorithms** - complexity analysis, efficient data handling

## Core Machine Learning

6. **Machine Learning Fundamentals** - supervised/unsupervised learning, bias-variance tradeoff
7. **Neural Network Basics** - perceptrons, activation functions, backpropagation
8. **Gradient Descent Variants** - SGD, Adam, learning rate schedules
9. **Regularization Techniques** - dropout, weight decay, batch/layer normalization
10. **Loss Functions** - cross-entropy, perplexity

## Deep Learning Architectures

11. **CNNs** (foundational understanding, even if n

In the following cell, the model is asked to produce a numbered list, but generation stops as soon as the sequence "4." appears (`stop_sequences`= [4]). This is a practical pattern when output needs to be concise or bounded.

In [17]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    stop_sequences=["4."],
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

# Technical Topics for Working on LLMs

## Foundations
1. Linear algebra (vectors, matrices, eigenvalues, tensor operations)
2. Probability and statistics (distributions, Bayesian thinking, entropy)
3. Calculus (gradients, chain rule, partial derivatives)



In [18]:
response

Message(id='msg_011Cfkdto5KsFneMZZ5U8RxB', content=[TextBlock(citations=None, text='# Technical Topics for Working on LLMs\n\n## Foundations\n1. Linear algebra (vectors, matrices, eigenvalues, tensor operations)\n2. Probability and statistics (distributions, Bayesian thinking, entropy)\n3. Calculus (gradients, chain rule, partial derivatives)\n', type='text')], model='claude-sonnet-5', role='assistant', stop_reason='stop_sequence', stop_sequence='4.', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=40, output_tokens=107, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

## Controlling response variability with prompting  (Controlling Model Behavior)

Earlier Claude models, such as `claude-3-5-sonnet-20241022`exposed sampling parameters such as `temperature`, which could be used to influence the variability of generated responses.

`Claude Sonnet 5` does not support non-default values for `temperature`, `top_p`, or `top_k`. Instead, **Anthropic recommends using prompting to guide the model's behavior**.

We can therefore experiment with the level of variation explicitly through instructions in the prompt.

### Prompting for consistency vs. creativity

In [19]:
def ask_claude(prompt):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=500,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.content[0].text

In [20]:
consistent_prompt = """
Write a two-sentence description of a deep learning model.

Be concise, factual, and consistent.
Use exactly two sentences.
Do not use figurative language.
"""

In [21]:
creative_prompt = """
Write a two-sentence description of a deep learning model.

Be creative and imaginative.
Use vivid language and different imagery.
Avoid generic phrases.
"""

In [22]:
print("=== CONSISTENT ===")
for _ in range(3):
    print(ask_claude(consistent_prompt))
    print()

print("=== CREATIVE ===")
for _ in range(3):
    print(ask_claude(creative_prompt))
    print()

=== CONSISTENT ===
A deep learning model is a computational system composed of multiple layers of interconnected nodes that process data to identify patterns and relationships. It learns these patterns by adjusting internal parameters through training on large datasets, enabling it to make predictions or classifications on new, unseen data.

A deep learning model is a computational system composed of multiple layers of interconnected nodes, or artificial neurons, that process data to identify patterns and relationships. It learns by adjusting the weights of connections between nodes through training on large datasets, enabling it to perform tasks such as classification, prediction, or generation.

A deep learning model is a computational system composed of multiple layers of interconnected nodes, or artificial neurons, that process data to identify patterns and relationships. It learns by adjusting the weights of connections between neurons through training on large datasets, enabling 

## Conclusion 

In this notebook, we learned how to:

- initialize the Anthropic client
- send requests through the Messages API
- structure user and assistant messages
- maintain conversation history
- build a simple chatbot
- control output length and stopping conditions
- handle response content safely

The most important concept is that **conversation context** is represented explicitly through the `messages` list. The application is responsible for deciding which previous messages to send with each request.

These patterns provide the foundation for the multimodal requests and tool-use workflows explored in the following notebooks.

## Practice challenges

This extra practice section is intended to help you become more confident with the Anthropic API 
and with prompt design, building your own AI applications. 


###  Challenge 1 — Conversation memory

Build a chatbot that remembers information provided earlier in the conversation.

**Goal:** Understand how message history provides context.


In [ ]:
## Snippet code example
messages = [
    {"role": "user", "content": "My favorite topic is Anthropic"},
    {"role": "assistant", "content": "Nice! I will remember that."}
]

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=200,
    messages=messages + [
        {"role": "user", "content": "Explain the most important discoveries in Anthropic."}
    ]
)

### Challenge 2 — Prompt comparison

Ask the same question using four different prompting styles such as:  
- direct question  
- detailed prompt  
- role-based prompt  
- step-by-step instruction  

**Goal:** Observe how instructions affect the response.


In [ ]:
## Snippet code example
prompts = [
    "Summarize machine learning in one paragraph.",
    "Explain machine learning to a beginner in simple language.",
    "Act as a teacher and explain machine learning with examples.",
    "Give me a short list of 5 key ideas in machine learning."
]

for prompt in prompts:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    print("PROMPT:", prompt)
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("\n" + "=" * 50 + "\n")

###  Challenge 3 — Structured output

Ask Claude to produce structured JSON and investigate how you could validate and parse the result in Python.

**Goal:** Move from text generation toward application-ready output.


In [ ]:
## Snippet code example
prompt = """
Return a JSON object with:
- name
- purpose
- benefits
- example_use_case

The answer must be valid JSON only.
"""

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

### Challenge 4 — Output limits

Run the same prompt with different `max_tokens` values and compare the results..

**Goal:** Understand how output limits affect generation.


In [ ]:
## Snippet code example
for tokens in [50, 150, 300]:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=tokens,
        messages=[{"role": "user", "content": "Write a short explanation of Anthropic"}]
    )
    print(f"MAX_TOKENS = {tokens}")
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("=" * 40)

###  Challenge 5 — Error handling

Experiment with invalid or unusual requests and inspect the resulting exceptions.

- empty prompts  
- very long prompts  
- very short outputs  
- ask for unsupported formatting  
- ask for a list with unusual constraints  

**Goal:** Learn how to build more resilient applications.    


In [ ]:
## Snippet code example  
try:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=50,
        messages=[{"role": "user", "content": ""}]
    )
except Exception as e:
    print("Error:", e)

###  Challenge 6 — Build your application 

Combine the concepts from this notebook to create a small application:

- study assistant
- writing assistant
- language tutor
- topic explainer
- Q&A bot



In [ ]:
## Snippet code example
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "You are a careful research assistant. Explain the difference between AI, ML, and deep learning in 3 bullet points."}
    ]
)
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

## What's next?

In this notebook, we worked primarily with text-based requests and conversation history.

In the **next notebook**, we will extend this foundation to **multimodal requests and streaming**, allowing Claude to work with visual inputs and produce responses incrementally.